# Training model Vine di Colab

Notebook ini melatih **IndoBERT** (butuh GPU) dan **model pembanding** (cepat), lalu membandingkannya.

Colab: menu **Runtime → Change runtime type → T4 GPU**.

Hasil akhirnya folder `models/` yang diunduh lalu ditaruh di laptop.

## 1. Ambil kode project

Pilih salah satu: clone dari GitHub tim (ganti URL), atau upload zip `ml-service` lewat panel Files lalu ekstrak.

In [ ]:
%cd ml-service


## 2. Pasang library

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas pyyaml pydantic-settings joblib


In [ ]:
import torch
print('GPU tersedia:', torch.cuda.is_available())


## 3. Siapkan data

Pembagian train/val/test dilakukan dulu, lalu hanya train yang diaugmentasi. Opsional: tambahkan data EmoT untuk emosi.

In [ ]:
!python training/prepare_data.py --task intent
!python training/prepare_data.py --task emotion


## 4. Model pembanding (beberapa detik)

Ini patokan. IndoBERT harus lebih baik dari angka ini.

In [ ]:
!python training/train_baseline.py
!python training/cross_validate.py --task intent
!python training/cross_validate.py --task emotion


## 5. IndoBERT maksud (alarm)

In [ ]:
!python training/train_intent.py --epoch 6
!python training/evaluate_intent.py --model indobert


Periksa **Recall alarm merah** (target ≥ 0.95) dan daftar **TERLEWAT**. Kalau belum lulus, coba `--ambang-darurat 0.2`, atau tambah kalimat darurat serupa ke `data/intent/raw/kalimat_pasien.csv` lalu latih ulang. Simpan ambang yang dipilih di `config/thresholds.yaml` → `intent.ambang_darurat`.

## 6. IndoBERT emosi

In [ ]:
!python training/train_emotion.py --epoch 6
!python training/evaluate_emotion.py --model indobert


Salin **ambang terbaik** ke `config/thresholds.yaml` → `emosi.ambang.indobert`.

## 7. Unduh hasil

In [ ]:
!zip -qr models_vine.zip models reports config/thresholds.yaml
from google.colab import files
files.download('models_vine.zip')


Di laptop: ekstrak `models_vine.zip` ke folder `ml-service` (timpa jika diminta), lalu nyalakan ulang server. Buka `/health` dan pastikan `intent` dan `emosi` bernilai `indobert`.